# Model Evaluation

## Objetivo

Evaluar y comparar el desempeño de los modelos desarrollados.

Se compararán:

1. Baseline basado en popularidad.
2. Collaborative Filtering User-Based.

La evaluación permitirá seleccionar el modelo final del sistema de recomendación.

In [1]:
# Importar librerías

import pandas as pd
import numpy as np

In [2]:
# Cargar dataset de modelado

model_data = pd.read_parquet(
    "../data/processed/model_data.parquet"
)

model_data.head()

,user_id,product_id,reordered,product_name,department,aisle
0,202279,33120,1,Organic Egg Whites,dairy eggs,eggs
1,202279,28985,1,Michigan Organic Kale,produce,fresh vegetables
2,202279,9327,0,Garlic Powder,pantry,spices seasonings
3,202279,45918,1,Coconut Butter,pantry,oils vinegars
4,202279,30035,0,Natural Sweetener,pantry,baking ingredients


In [3]:
# Usuarios con al menos 20 productos distintos

evaluation_users = (
    model_data.groupby("user_id")["product_id"]
    .nunique()
    .reset_index()
)

evaluation_users = evaluation_users[
    evaluation_users["product_id"] >= 20
]

len(evaluation_users)

136688

In [4]:
# Muestra reproducible

evaluation_users = (
    evaluation_users["user_id"]
    .sample(
        n=500,
        random_state=42
    )
)

len(evaluation_users)

500

In [5]:
# Crear conjunto de prueba

test_items = []

for user in evaluation_users:

    user_products = model_data[
        model_data["user_id"] == user
    ]["product_id"].unique()

    hidden_product = np.random.choice(user_products)

    test_items.append(
        {
            "user_id": user,
            "hidden_product": hidden_product
        }
    )

test_df = pd.DataFrame(test_items)

test_df.head()

,user_id,hidden_product
0,178480,42830
1,192973,22210
2,96149,24799
3,98836,6473
4,193278,48720


In [6]:
# Ranking global de productos

popular_products = (
    model_data
    .groupby("product_id")
    .size()
    .reset_index(name="purchase_count")
    .sort_values(
        by="purchase_count",
        ascending=False
    )
)

top10_products = set(
    popular_products
    .head(10)["product_id"]
)

In [7]:
# Calcular Hit Rate del Baseline

hits = 0

for product in test_df["hidden_product"]:

    if product in top10_products:
        hits += 1

baseline_hit_rate = hits / len(test_df)

baseline_hit_rate

0.046

# Evaluación del Baseline

## Métrica utilizada

Hit Rate@10

La métrica evalúa si el producto ocultado durante la validación aparece dentro de las 10 recomendaciones generadas por el modelo.

## Resultado

Hit Rate@10 = 4,6%

## Interpretación

El modelo basado en popularidad logra identificar correctamente el producto ocultado en aproximadamente 5 de cada 100 casos.

Si bien constituye una referencia simple y eficiente, presenta limitaciones importantes al no considerar preferencias individuales de los usuarios.

In [8]:
# Seleccionar 100 usuarios para evaluación

evaluation_users_100 = (
    test_df["user_id"]
    .sample(
        n=100,
        random_state=42
    )
)

len(evaluation_users_100)

100

In [9]:
# Diccionario user -> hidden product

hidden_products = dict(
    zip(
        test_df["user_id"],
        test_df["hidden_product"]
    )
)

In [10]:
# Evaluar si el producto ocultado aparece en las recomendaciones

def hit_at_10(user_id):

    hidden_product = hidden_products[user_id]

    try:

        recommendations = recommend_user_based(
            user_id,
            n_recommendations=10
        )

        recommended_products = set(
            recommendations["product_id"]
        )

        return int(
            hidden_product in recommended_products
        )

    except:
        return 0

In [11]:
# Evaluar Collaborative Filtering

hits = []

for user in evaluation_users_100:

    hits.append(
        hit_at_10(user)
    )

cf_hit_rate = np.mean(hits)

cf_hit_rate

np.float64(0.0)

In [12]:
# Comparación de modelos

results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Collaborative Filtering"
    ],
    "HitRate@10": [
        baseline_hit_rate,
        cf_hit_rate
    ]
})

results

,Model,HitRate@10
0,Baseline,0.046
1,Collaborative Filtering,0.000


# Reconstrucción del conjunto de entrenamiento

## Problema detectado

Durante la evaluación inicial se observó que el modelo de Collaborative Filtering obtenía un Hit Rate@10 igual a 0%.

Al revisar la metodología se identificó que la matriz usuario-producto había sido construida utilizando el historial completo de los usuarios, incluyendo los productos posteriormente utilizados para evaluación.

Esto genera una fuga de información (Data Leakage), ya que el modelo tiene acceso indirecto a información que debería permanecer oculta durante la validación.

## Solución

Para corregir este problema se construirá un nuevo conjunto de entrenamiento eliminando previamente los productos ocultados para cada usuario.

Posteriormente se reconstruirán:

- La matriz usuario-producto.
- La matriz de similitud entre usuarios.
- El sistema de recomendación.

De esta forma la evaluación reflejará de manera más realista la capacidad del modelo para recomendar productos no observados durante el entrenamiento.

In [13]:
# Crear copia del dataset

train_data = model_data.copy()

In [15]:
# Crear identificador auxiliar para remover interacciones ocultadas de forma eficiente

model_data_eval = model_data.copy()

model_data_eval["user_product_key"] = (
    model_data_eval["user_id"].astype(str)
    + "_"
    + model_data_eval["product_id"].astype(str)
)

test_df["user_product_key"] = (
    test_df["user_id"].astype(str)
    + "_"
    + test_df["hidden_product"].astype(str)
)

In [16]:
# Crear conjunto de entrenamiento eliminando productos ocultados

train_data = model_data_eval[
    ~model_data_eval["user_product_key"].isin(test_df["user_product_key"])
].copy()

train_data = train_data.drop(columns=["user_product_key"])

In [17]:
# Verificar tamaño del conjunto de entrenamiento

print("Dataset original:", model_data.shape)
print("Dataset entrenamiento:", train_data.shape)
print("Filas eliminadas:", model_data.shape[0] - train_data.shape[0])

Dataset original: (30431675, 6)
Dataset entrenamiento: (30430383, 6)
Filas eliminadas: 1292


In [18]:
train_data.shape

(30430383, 6)

In [19]:
sample_users = (
    train_data["user_id"]
    .drop_duplicates()
    .sample(
        n=10000,
        random_state=42
    )
)

In [20]:
cf_train_data = train_data[
    train_data["user_id"].isin(sample_users)
].copy()

In [21]:
cf_train_data["interaction"] = 1

user_product_matrix = cf_train_data.pivot_table(
    index="user_id",
    columns="product_id",
    values="interaction",
    fill_value=0
)

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(
    user_product_matrix
)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_product_matrix.index,
    columns=user_product_matrix.index
)

In [23]:
evaluation_users_in_matrix = [
    u
    for u in evaluation_users
    if u in user_product_matrix.index
]

len(evaluation_users_in_matrix)

32

# Ajuste de la muestra para evaluación

Se observó que una muestra aleatoria de usuarios podía excluir gran parte de los usuarios seleccionados para validación.

Para garantizar una evaluación consistente, se reconstruirá la muestra incluyendo explícitamente los usuarios utilizados en el conjunto de prueba.

De esta manera se asegura que todos los usuarios evaluados estén presentes en la matriz usuario-producto y puedan recibir recomendaciones.

# Evaluación corregida

Luego de eliminar los productos ocultados del conjunto de entrenamiento se reconstruyó completamente el modelo de Collaborative Filtering.

Este procedimiento evita la fuga de información y permite obtener métricas de desempeño más representativas del comportamiento esperado en producción.

In [25]:
# Convertir usuarios de evaluación a conjunto

evaluation_users_set = set(evaluation_users)

len(evaluation_users_set)

500

In [26]:
# Seleccionar usuarios adicionales

additional_users = (
    train_data[
        ~train_data["user_id"].isin(evaluation_users_set)
    ]["user_id"]
    .drop_duplicates()
    .sample(
        n=9500,
        random_state=42
    )
)

len(additional_users)

9500

In [27]:
# Construir muestra final

sample_users = pd.concat([
    pd.Series(list(evaluation_users_set)),
    additional_users
]).unique()

len(sample_users)

10000

In [28]:
# Reconstruir dataset de entrenamiento

cf_train_data = train_data[
    train_data["user_id"].isin(sample_users)
].copy()

cf_train_data.shape

(2186525, 6)

In [29]:
# Crear interacción

cf_train_data["interaction"] = 1

In [30]:
# Reconstruir matriz usuario-producto

user_product_matrix = cf_train_data.pivot_table(
    index="user_id",
    columns="product_id",
    values="interaction",
    fill_value=0
)

user_product_matrix.shape

(10000, 32566)

In [31]:
# Reconstruir similitud

from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(
    user_product_matrix
)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_product_matrix.index,
    columns=user_product_matrix.index
)

In [32]:
evaluation_users_in_matrix = [
    u
    for u in evaluation_users
    if u in user_product_matrix.index
]

len(evaluation_users_in_matrix)

500

In [33]:
# Función corregida para recomendar productos usando usuarios similares

def recommend_user_based_eval(user_id, n_recommendations=10, n_similar_users=10):
    
    similar_users = (
        user_similarity_df[user_id]
        .sort_values(ascending=False)
        .iloc[1:n_similar_users + 1]
        .index
    )
    
    user_products = set(
        user_product_matrix.loc[user_id][
            user_product_matrix.loc[user_id] > 0
        ].index
    )
    
    similar_users_products = user_product_matrix.loc[similar_users]
    
    product_scores = (
        similar_users_products
        .sum(axis=0)
        .sort_values(ascending=False)
    )
    
    recommendations = product_scores[
        ~product_scores.index.isin(user_products)
    ].head(n_recommendations)
    
    return list(recommendations.index)

In [34]:
# Calcular Hit Rate@10 para Collaborative Filtering

hits = []

for user in evaluation_users_in_matrix:
    
    hidden_product = hidden_products[user]
    
    recommended_products = recommend_user_based_eval(
        user,
        n_recommendations=10
    )
    
    hits.append(
        int(hidden_product in recommended_products)
    )

cf_hit_rate = np.mean(hits)

cf_hit_rate

np.float64(0.066)

In [35]:
# Comparar resultados de los modelos

results = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Collaborative Filtering User-Based"
    ],
    "HitRate@10": [
        baseline_hit_rate,
        cf_hit_rate
    ]
})

results

,Modelo,HitRate@10
0,Baseline,0.046
1,Collaborative Filtering User-Based,0.066


# Comparación de modelos

## Resultados

| Modelo | Hit Rate@10 |
|---------|------------|
| Baseline | 4,6% |
| Collaborative Filtering User-Based | 6,6% |

## Interpretación

El modelo de Collaborative Filtering obtuvo un mejor desempeño que el Baseline basado en popularidad.

Mientras que el Baseline logra recuperar el producto ocultado en aproximadamente 4,6 de cada 100 usuarios, el modelo colaborativo alcanza 6,6 de cada 100 usuarios.

Esto representa una mejora aproximada del 43,5% respecto del modelo de referencia.

## Conclusión

Los resultados sugieren que incorporar información sobre similitud entre usuarios permite generar recomendaciones más relevantes que una estrategia basada únicamente en popularidad.

Por este motivo se selecciona Collaborative Filtering User-Based como modelo final del sistema de recomendación.